# AI Scientist CLOP Pipeline — Colab Quickstart

This notebook runs the **sequence→annotation AI Scientist** pipeline on consumer hardware.
You will:
1. Install deps and the pipeline from a local ZIP (upload),
2. (Option A) Run CLOP quickly on a tiny toy set (1 epoch), **or**
3. (Option B, fastest) Use a tiny precomputed embedding table for demo,
4. Generate hypotheses → build **annotation graphs** → (optionally) validate → report.


In [ ]:
# 1) Install dependencies
!pip -q install biopython pandas numpy scikit-learn typer rich jinja2 networkx pyarrow faiss-cpu || pip -q install faiss-cpu


In [ ]:
from google.colab import files
up = files.upload()
zip_name = next(iter(up.keys()))
import os, zipfile, shutil, sys
work = "/content/work"
os.makedirs(work, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(work)
%cd /content/work/zoeva
!pip -q install -e .


In [ ]:
!git clone -q https://github.com/baudrly/clop /content/clop || true
clop_repo = "/content/clop"


In [ ]:
fasta = "/content/work/data.fasta"
open(fasta, "w").write(">seq1|species=Escherichia coli|biotype=CDS|description=phosphoglycerate kinase glycolysis\nATGGCGCGCGCGCGCTAG\n>seq2|species=Bacillus subtilis|biotype=CDS|description=mitochondrion ribosomal protein translation\nATGCGTATATATATATATA\n>seq3|species=Saccharomyces cerevisiae|biotype=CDS|description=dehydrogenase TCA cycle\nATGCATGCATGCATGCATG\n")
print(open(fasta).read())


In [ ]:
#Run CLOP quickly (k=4, epochs=1) to export embeddings
!zoeva embed --clop_repo /content/clop --input /content/work/data.fasta --input-type fasta --out /content/work/run/embeddings --kmer_k 4 --epochs 1 --batch_size 8 || echo "If CLOP fails or takes too long, skip to Option B below."


In [ ]:
# 5b) Option B (fastest): create a tiny placeholder embedding table to demo downstream steps
import pandas as pd, numpy as np, os
emb = pd.DataFrame({
    "id": ["seq1","seq2","seq3"],
    **{f"e{i}": v for i,v in enumerate(np.stack([
        np.array([0.9,0.1,0.0,0.2,0.1,0.0,0.1,0.0]),
        np.array([0.2,0.9,0.1,0.0,0.1,0.0,0.0,0.1]),
        np.array([0.1,0.1,0.9,0.0,0.2,0.1,0.0,0.0])
    ]))}
})
os.makedirs("/content/work/run/embeddings", exist_ok=True)
emb.to_csv("/content/work/run/embeddings/final_embeddings.csv", index=False)
print(emb.head())


In [ ]:
# 6) Hypothesis generation (kNN + optional zero-shot)
!zoeva hypothesize --embeddings /content/work/run/embeddings/final_embeddings.csv --out /content/work/run/hypotheses


In [ ]:
# 7) Build annotation graph & run personalized PageRank
!zoeva graph --hypotheses /content/work/run/hypotheses/hypotheses.parquet --out /content/work/run/graph --kw_to_label /content/work/zoeva/ref/kw_to_label.tsv --label_edges /content/work/zoeva/ref/label_edges.tsv


In [ ]:
# 8) (Optional) Classical validators 
# !zoeva validate --fasta /content/work/data.fasta --out /content/work/run/validation --pfam_hmm /content/Pfam-A.hmm


In [ ]:
# 9) Score & report (including graph evidence)
!zoeva report --hypotheses /content/work/run/hypotheses/hypotheses.parquet --graph /content/work/run/graph/graph_scores.parquet --out /content/work/run/report
print(open("/content/work/run/report/report.md").read())
